### Fine-tune Whisper
- Fine-tuning websites: https://github.com/Vaibhavs10/fast-whisper-finetuning/tree/main; https://huggingface.co/docs/peft/main/en/task_guides/int8-asr; https://huggingface.co/blog/fine-tune-whisper; https://learnopencv.com/fine-tuning-whisper-on-custom-dataset/
- Pre-trained model: **openai/whisper-large-v3**
- They claim as little as 8 hours of fine-tuning data, we can achieve strong performance.
- Test code by fine-tuning on yue: https://huggingface.co/datasets/mozilla-foundation/common_voice_11_0/viewer/yue, using **openai/whisper-small**

In [48]:
!pip install peft git+https://github.com/huggingface/peft@ec267c6

  Cloning https://github.com/huggingface/peft (to revision ec267c6) to /tmp/pip-req-build-_0_9iy7p
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/peft /tmp/pip-req-build-_0_9iy7p
  Running command git checkout -q ec267c6
  Resolved https://github.com/huggingface/peft to commit ec267c6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for peft: filename=peft-0.5.0.dev0-py3-none-any.whl size=73952 sha256=b262ca974c80a2d9eccef3b450a0900a7fc3a96e24227e0859decbf44593bb82
  Stored in directory: /tmp/pip-ephem-wheel-cache-qgtl5wty/wheels/c8/a3/50/4168baa068c6475947e49129930a2e16361f03f703e2057ea9
Successfully built peft


In [25]:
!pip install --upgrade pip
!pip install --upgrade datasets[audio] transformers accelerate evaluate jiwer bitsandbytes==0.37 accelerate
!pip install -q git+https://github.com/huggingface/peft.git@main
!pip install wandb

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 97.8 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade pip
!pip install --upgrade datasets[audio] transformers accelerate evaluate jiwer bitsandbytes accelerate
!pip install -q git+https://github.com/huggingface/peft.git@main
!pip install wandb

In [38]:
!pip uninstall -y bitsandbytes

Found existing installation: bitsandbytes 0.37.0
Uninstalling bitsandbytes-0.37.0:
  Successfully uninstalled bitsandbytes-0.37.0


In [40]:
# cuda 122 - not compatible with current bitsandbbytes
!git clone https://github.com/timdettmers/bitsandbytes.git
!cd bitsandbytes; CUDA_VERSION=122 make cuda12x; python setup.py install

fatal: destination path 'bitsandbytes' already exists and is not an empty directory.
make: *** No rule to make target 'cuda12x'.  Stop.
libs: []
/usr/local/lib/python3.10/dist-packages/setuptools/dist.py:300: InformationOnly: Normalizing '0.44.2.dev' to '0.44.2.dev0'
  self.metadata.version = self._normalize_version(self.metadata.version)
running install
/usr/local/lib/python3.10/dist-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()
/usr/local/lib/python3.10/dist-packages/setuptools/_distutils/

In [27]:
!wandb login

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [2]:
from huggingface_hub import notebook_login

notebook_login()

In [1]:
from datasets import load_dataset, DatasetDict

common_voice = DatasetDict()

common_voice["train"] = load_dataset("mozilla-foundation/common_voice_11_0", "yue", split="train+validation")
common_voice["test"] = load_dataset("mozilla-foundation/common_voice_11_0", "yue", split="test")
common_voice = common_voice.remove_columns(["accent", "age", "client_id", "down_votes", "gender", "locale", "path", "segment", "up_votes"])

print(common_voice)
print(common_voice["train"][0])

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 5296
    })
    test: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 2438
    })
})
{'audio': {'path': '/root/.cache/huggingface/datasets/downloads/extracted/f0f4ec3cfc702df3c79a6a69dc787b516e21f2c0f0af23da91713af8ade367b1/yue_train_0/common_voice_yue_31209989.mp3', 'array': array([ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
       -7.54986104e-06, -5.66183462e-06, -1.75592686e-06]), 'sampling_rate': 48000}, 'sentence': '美國都係唔考慮喇，睇下澳洲先'}


In [ ]:
# A feature extractor which pre-processes the raw audio-inputs
# Whisper feature extractor expects audio inputs with a sampling rate of 16kHz
# It first pads/truncates a batch of audio samples such that all samples have an input length of 30s.
# The second operation that the Whisper feature extractor performs is converting the padded audio arrays to log-Mel spectrograms.
# The log-Mel spectrogram is the form of input expected by the Whisper model.

In [ ]:
# A tokenizer which post-processes the model outputs to text format
# These arguments inform the tokenizer to prefix the target language and task tokens to the start of encoded label sequences:

In [5]:
# To simplify using the feature extractor and tokenizer, we can wrap both into a single WhisperProcessor class.

from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="zh", task="transcribe") # zh

In [15]:
processor.feature_extractor

WhisperFeatureExtractor {
  "chunk_length": 30,
  "feature_extractor_type": "WhisperFeatureExtractor",
  "feature_size": 80,
  "hop_length": 160,
  "n_fft": 400,
  "n_samples": 480000,
  "nb_max_frames": 3000,
  "padding_side": "right",
  "padding_value": 0.0,
  "processor_class": "WhisperProcessor",
  "return_attention_mask": false,
  "sampling_rate": 16000
}

In [16]:
processor.tokenizer

WhisperTokenizer(name_or_path='openai/whisper-small', vocab_size=50258, model_max_length=1024, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|endoftext|>', '<|startoftranscript|>', '<|en|>', '<|zh|>', '<|de|>', '<|es|>', '<|ru|>', '<|ko|>', '<|fr|>', '<|ja|>', '<|pt|>', '<|tr|>', '<|pl|>', '<|ca|>', '<|nl|>', '<|ar|>', '<|sv|>', '<|it|>', '<|id|>', '<|hi|>', '<|fi|>', '<|vi|>', '<|he|>', '<|uk|>', '<|el|>', '<|ms|>', '<|cs|>', '<|ro|>', '<|da|>', '<|hu|>', '<|ta|>', '<|no|>', '<|th|>', '<|ur|>', '<|hr|>', '<|bg|>', '<|lt|>', '<|la|>', '<|mi|>', '<|ml|>', '<|cy|>', '<|sk|>', '<|te|>', '<|fa|>', '<|lv|>', '<|bn|>', '<|sr|>', '<|az|>', '<|sl|>', '<|kn|>', '<|et|>', '<|mk|>', '<|br|>', '<|eu|>', '<|is|>', '<|hy|>', '<|ne|>', '<|mn|>', '<|bs|>', '<|kk|>', '<|sq|>', '<|sw|>', '<|gl|>', '<|mr|>', '<|pa|>', '<|si

In [17]:
# testing tokenizer
input_str = common_voice["train"][0]["sentence"]
labels = processor.tokenizer(input_str).input_ids
decoded_with_special = processor.tokenizer.decode(labels, skip_special_tokens=False)
decoded_str = processor.tokenizer.decode(labels, skip_special_tokens=True)

print(f"Input:                 {input_str}")
print(f"Decoded w/ special:    {decoded_with_special}")
print(f"Decoded w/out special: {decoded_str}")
print(f"Are equal:             {input_str == decoded_str}")

Input:                 美國都係唔考慮喇，睇下澳洲先
Decoded w/ special:    <|startoftranscript|><|zh|><|transcribe|><|notimestamps|>美國都係唔考慮喇，睇下澳洲先<|endoftext|>
Decoded w/out special: 美國都係唔考慮喇，睇下澳洲先
Are equal:             True


In [6]:
# data preparation
from datasets import Audio

# input audio is sampled at 48kHz, we need to downsample it to 16kHz
common_voice = common_voice.cast_column("audio", Audio(sampling_rate=16000))
# resample, get log mel, tokenize
common_voice = common_voice.map(prepare_dataset, remove_columns=common_voice.column_names["train"], num_proc=None) # 4

#### Training Functions:

In [8]:
import evaluate, torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers.trainer_utils import PREFIX_CHECKPOINT_DIR
from transformers import TrainerCallback, TrainingArguments, TrainerState, TrainerControl

# Data Preparation: resample, get log mel, tokenize
processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="zh", task="transcribe") ### zh
def prepare_dataset(batch):
    # load and resample audio data from 48 to 16kHz
    audio = batch["audio"]

    # compute log-Mel input features from input audio array
    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    # encode target text to label ids
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

# Evaluation Metrics
metric = evaluate.load("wer")
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

# Data collator
# input_features are already padded, just need to batch them
# labels are unpadded, we need to pad
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

# It is also a good idea to write a custom TrainerCallback to save model checkpoints during training:
class SavePeftModelCallback(TrainerCallback):
    def on_save(
        self,
        args: TrainingArguments,
        state: TrainerState,
        control: TrainerControl,
        **kwargs,
    ):
        checkpoint_folder = os.path.join(args.output_dir, f"{PREFIX_CHECKPOINT_DIR}-{state.global_step}")

        peft_model_path = os.path.join(checkpoint_folder, "adapter_model")
        kwargs["model"].save_pretrained(peft_model_path)

        pytorch_model_path = os.path.join(checkpoint_folder, "pytorch_model.bin")
        if os.path.exists(pytorch_model_path):
            os.remove(pytorch_model_path)
        return control

#### Training script

In [31]:
# don't work
!apt-get update
!apt-get install cuda-toolkit-11-8
import os
os.environ["LD_LIBRARY_PATH"] += ":" + "/usr/local/cuda-11/lib64"
os.environ["LD_LIBRARY_PATH"] += ":" + "/usr/local/cuda-11.8/lib64"


0% [Working]
            
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,031 kB]
Ign:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy Release [5,713 B]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,161 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy Release.gpg [793 B]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [2,326 kB]


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, Seq2SeqTrainingArguments
from peft import LoraConfig, PeftModel, LoraModel, LoraConfig, get_peft_model
import wandb

# start a new wandb run to track this script
wandb.init(
    # set the wandb project where this run will be logged
    project="test_whisper_finetune",
    # # track hyperparameters and run metadata
    # config={
    # "learning_rate": 0.02,
    # "architecture": "CNN",
    # "dataset": "CIFAR-100",
    # "epochs": 10,
    # }
)

# Data preparation
###

processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="zh", task="transcribe") ### zh

# Load a Pre-Trained Checkpoint
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small", device_map="auto") # load_in_8bit=True
model.generation_config.language = "zh" ###
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
# In cases where the source audio language is known a-priori, such as multilingual fine-tuning, it is beneficial to set the language explicitly.
# This negates the scenarios when the incorrect language is predicted, causing the predicted text to diverge from the true language during generation.

# combine pre-trained model (freezed) w PEFT (trainable)
config = LoraConfig(r=32, lora_alpha=64, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none")
model = get_peft_model(model, config)
model.print_trainable_parameters()

# Data collator
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

# Define the Training Configuration
# The PeftModel doesn’t have the same signature as the base model, so you’ll need to explicitly set remove_unused_columns=False and label_names=["labels"].
training_args = Seq2SeqTrainingArguments(
    seed = 2024, ###
    # learning_rate=2e-5, ###
    # optim = "paged_adamw_8bit", ###
    output_dir="finetuned", ###
    per_device_train_batch_size=16, ###
    gradient_accumulation_steps=1,
    learning_rate=1e-3,
    # warmup_steps=10, ###
    warmup_ratio=0.01, ###
    num_train_epochs=3,
    do_eval=True,
    eval_steps=10, ###
    eval_strategy="steps", # epoch
    max_steps=60, ###
    fp16=True, ### bf16=True
    per_device_eval_batch_size=16, ###
    predict_with_generate=True,
    generation_max_length=128,
    logging_strategy="steps",
    logging_steps=10, ###
    remove_unused_columns=False,
    label_names=["labels"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    save_strategy = "steps",
    save_steps = 30, ###
    save_total_limit = 2, ###
)

# Define trainer
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    callbacks=[SavePeftModelCallback],
)
model.config.use_cache = False  # silence the warnings, reanable for inference
trainer.train()

train/epoch,▁
train/global_step,▁
train/grad_norm,▁
train/learning_rate,▁
train/loss,▁
train/epoch,0.03021
train/global_step,10
train/grad_norm,1.19145
train/learning_rate,0.001
train/loss,0.81


max_steps is given, it will override any value given in num_train_epochs


trainable params: 3,538,944 || all params: 245,273,856 || trainable%: 1.442854145857274


Step,Training Loss,Validation Loss


#### Evaluation

In [ ]:
# Loading trained model
from peft import PeftModel, PeftConfig
from transformers import WhisperForConditionalGeneration, Seq2SeqTrainer

peft_model_id = "finetuned" ###
peft_config = PeftConfig.from_pretrained(peft_model_id)
model = WhisperForConditionalGeneration.from_pretrained(
    peft_config.base_model_name_or_path, device_map="auto"
)
model = PeftModel.from_pretrained(model, peft_model_id)
model.config.use_cache = True

In [ ]:
# Evaluation on a test set (MAYBE CAN NORMALIZE TEXT BEFORE EVALUATION???)
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
import gc

eval_dataloader = DataLoader(common_voice["test"], batch_size=8, collate_fn=data_collator)
forced_decoder_ids = processor.get_decoder_prompt_ids(language="yue", task="transcribe")

# to output later on
predictions = []
references = []

model.eval()
for step, batch in enumerate(tqdm(eval_dataloader)):
    # with torch.cuda.amp.autocast():
        with torch.no_grad():
            generated_tokens = (
                model.generate(
                    input_features=batch["input_features"].to("cuda"),
                    # decoder_input_ids=batch["labels"][:, :4].to("cuda"),
                    forced_decoder_ids=forced_decoder_ids,
                    max_new_tokens=255,
                )
                .cpu()
                .numpy()
            )
            labels = batch["labels"].cpu().numpy()
            labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
            decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
            decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
            predictions.extend(decoded_preds)
            references.extend(decoded_labels)
            metric.add_batch(
                predictions=decoded_preds,
                references=decoded_labels,
            )
    del generated_tokens, labels, batch
    gc.collect()
wer = 100 * metric.compute()
print(f"{wer=}")

In [ ]:
# Inference with Pipeline
from transformers import (
    AutomaticSpeechRecognitionPipeline,
    WhisperForConditionalGeneration,
    WhisperTokenizer,
    WhisperProcessor,
)
from peft import PeftModel, PeftConfig

peft_model_id = "finetuned" ###
language = "yue"
task = "transcribe"
peft_config = PeftConfig.from_pretrained(peft_model_id)
model = WhisperForConditionalGeneration.from_pretrained(
    peft_config.base_model_name_or_path, device_map="auto"
)
model = PeftModel.from_pretrained(model, peft_model_id)
tokenizer = WhisperTokenizer.from_pretrained(peft_config.base_model_name_or_path, language=language, task=task)
processor = WhisperProcessor.from_pretrained(peft_config.base_model_name_or_path, language=language, task=task)
feature_extractor = processor.feature_extractor
forced_decoder_ids = processor.get_decoder_prompt_ids(language=language, task=task)

pipeline = AutomaticSpeechRecognitionPipeline(model=model, tokenizer=tokenizer, feature_extractor=feature_extractor)

# audio to test
audio =
text = pipe(audio, generate_kwargs={"forced_decoder_ids": forced_decoder_ids}, max_new_tokens=255)["text"]
print(text)